# Feature engineering improvement search

Two-phase Optuna hyperparameter search for XGBoost, comparing two feature-engineering strategies head-to-head:

1. **Phase 1** — feature engineering + target encoding pipeline: all 12 raw columns cast to `category`, plus the 10 engineered ratio features, plus out-of-fold target/frequency encoding (`_te`/`_freq`) for every raw column (46 features total).
2. **Phase 2** — native dtypes: numeric columns stay numeric, the 3 nominal columns stay `category`, plus the same 10 engineered ratio features. No `_te`/`_freq` encoding (22 features total).

Each phase gets its own wall-clock time budget (set via `MAX_SECONDS_PHASE1`/`MAX_SECONDS_PHASE2`). Set `QUICK_TEST = True` to run both phases for ~2 minutes locally just to confirm everything works before kicking off the full 3h/3h run on Kaggle.

Uses Optuna (TPE sampler + median pruning) instead of plain random search: it reuses information from earlier trials to bias later ones, and pruning kills clearly bad trials early — both help more within a hard time budget than blind random sampling.

In [1]:
import os
import time
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

try:
    import optuna
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
# ===== Config =====
SEED = 42
N_SPLITS = 3
EARLY_STOPPING_ROUNDS = 50

QUICK_TEST = True  # set False for the real Kaggle run
MAX_SECONDS_PHASE1 = 120 if QUICK_TEST else 3 * 3600
MAX_SECONDS_PHASE2 = 120 if QUICK_TEST else 3 * 3600

print(f"QUICK_TEST={QUICK_TEST} | phase1 budget={MAX_SECONDS_PHASE1}s | phase2 budget={MAX_SECONDS_PHASE2}s")

QUICK_TEST=True | phase1 budget=120s | phase2 budget=120s


In [3]:
# ===== Data path auto-detect (local vs Kaggle) =====
local_train = "/workspaces/Kaggle/KagglePredictPhoneAddiction/data/train.csv"
kaggle_train = "/kaggle/input/competitions/playground-series-s6e8/train.csv"
train_path = local_train if os.path.exists(local_train) else kaggle_train
if not os.path.exists(train_path):
    raise FileNotFoundError("Could not find train.csv in local or Kaggle path.")

print("Train file:", train_path)
df = pd.read_csv(train_path)

TARGET = "addicted_label"
ID_COL = "id" if "id" in df.columns else None

out_dir = "/kaggle/working/feature_eng_improvement_outputs" if os.path.exists("/kaggle/working") else "/workspaces/Kaggle/KagglePredictPhoneAddiction/feature_eng_improvement_outputs"
os.makedirs(out_dir, exist_ok=True)
print("Output dir:", out_dir)

y = df[TARGET].astype(int)

Train file: /workspaces/Kaggle/KagglePredictPhoneAddiction/data/train.csv


Output dir: /workspaces/Kaggle/KagglePredictPhoneAddiction/feature_eng_improvement_outputs


In [4]:
# ===== Shared column groups =====
numeric_cols = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time",
]
categorical_cols = ["gender", "stress_level", "academic_work_impact"]
initial_features = numeric_cols + categorical_cols

# ===== Phase 1: feature engineering + target encoding =====
# Group 1 (all 12 raw cols cast to category) + Group 2 (10 engineered ratios) + Group 3 (te/freq per column)
df_p1 = df.copy()
for col in initial_features:
    df_p1[col] = df_p1[col].astype("str").astype("category")

target_enc_cols = list(initial_features)

def engineer_features(source_df):
    source_df = source_df.copy()

    source_df["missing_count"] = source_df[initial_features].apply(
        lambda row: row.astype(str).isin(["nan", "None", "<NA>"]).sum(), axis=1
    )

    def to_float(col_name):
        if col_name in source_df.columns:
            return pd.to_numeric(source_df[col_name].astype(str), errors="coerce")
        return pd.Series(np.nan, index=source_df.index)

    s_media = to_float("social_media_hours")
    gaming = to_float("gaming_hours")
    work = to_float("work_study_hours")
    daily_screen = to_float("daily_screen_time_hours")
    sleep = to_float("sleep_hours")
    weekend_screen = to_float("weekend_screen_time")
    app_opens = to_float("app_opens_per_day")
    notifications = to_float("notifications_per_day")

    source_df["total_breakdown_hours"] = s_media.fillna(0) + gaming.fillna(0) + work.fillna(0)
    source_df["social_ratio"] = s_media / (daily_screen + 1e-5)
    source_df["gaming_ratio"] = gaming / (daily_screen + 1e-5)
    source_df["work_ratio"] = work / (daily_screen + 1e-5)
    source_df["unaccounted_screen_time"] = daily_screen - source_df["total_breakdown_hours"]
    source_df["screen_to_sleep_ratio"] = daily_screen / (sleep + 1e-5)
    source_df["weekend_vs_daily_ratio"] = weekend_screen / (daily_screen + 1e-5)
    source_df["app_opens_per_hour"] = app_opens / (daily_screen + 1e-5)
    source_df["notifications_per_hour"] = notifications / (daily_screen + 1e-5)

    return source_df

df_p1 = engineer_features(df_p1)

def apply_encodings_cv(train_df, test_df, cat_cols, target_col, n_splits=5, smoothing=10, random_state=42):
    """Out-of-fold target + frequency encoding, identical to the feature engineering + target encoding approach.
    test_df has no effect on the returned train_encoded; kept only for signature
    fidelity since this search notebook has no separate Kaggle test set to encode."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    train_encoded = train_df.copy()
    test_encoded = test_df.copy()

    for col in cat_cols:
        train_encoded[f"{col}_te"] = 0.0
        train_encoded[f"{col}_freq"] = 0.0
        test_encoded[f"{col}_freq"] = 0.0
        test_encoded[f"{col}_te"] = 0.0

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df[target_col])):
        tr_subset = train_df.iloc[train_idx]
        val_subset = train_df.iloc[val_idx]
        global_mean = float(tr_subset[target_col].mean())

        for col in cat_cols:
            counts = tr_subset[col].value_counts()

            val_freq = val_subset[col].map(counts).astype(float).fillna(0.0).values
            train_encoded.iloc[val_idx, train_encoded.columns.get_loc(f"{col}_freq")] = val_freq

            test_freq = test_df[col].map(counts).astype(float).fillna(0.0).values
            test_encoded[f"{col}_freq"] += test_freq / n_splits

            stats = tr_subset.groupby(col, observed=False)[target_col].agg(["count", "mean"])
            smoothed_te = (stats["count"] * stats["mean"] + smoothing * global_mean) / (stats["count"] + smoothing)

            val_te = val_subset[col].map(smoothed_te).astype(float).fillna(global_mean).values
            train_encoded.iloc[val_idx, train_encoded.columns.get_loc(f"{col}_te")] = val_te

            test_te = test_df[col].map(smoothed_te).astype(float).fillna(global_mean).values
            test_encoded[f"{col}_te"] += test_te / n_splits

    return train_encoded, test_encoded, skf

df_p1_encoded, _, _ = apply_encodings_cv(df_p1, df_p1, cat_cols=target_enc_cols, target_col=TARGET)

feature_cols_phase1 = [c for c in df_p1_encoded.columns if c not in [TARGET, ID_COL]]
X_phase1 = df_p1_encoded[feature_cols_phase1].copy()
categorical_cols_phase1 = target_enc_cols  # all 12 raw columns are category dtype

print(f"Phase 1 features ({len(feature_cols_phase1)}):", feature_cols_phase1)

Phase 1 features (46): ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact', 'missing_count', 'total_breakdown_hours', 'social_ratio', 'gaming_ratio', 'work_ratio', 'unaccounted_screen_time', 'screen_to_sleep_ratio', 'weekend_vs_daily_ratio', 'app_opens_per_hour', 'notifications_per_hour', 'age_te', 'age_freq', 'daily_screen_time_hours_te', 'daily_screen_time_hours_freq', 'social_media_hours_te', 'social_media_hours_freq', 'gaming_hours_te', 'gaming_hours_freq', 'work_study_hours_te', 'work_study_hours_freq', 'sleep_hours_te', 'sleep_hours_freq', 'notifications_per_day_te', 'notifications_per_day_freq', 'app_opens_per_day_te', 'app_opens_per_day_freq', 'weekend_screen_time_te', 'weekend_screen_time_freq', 'gender_te', 'gender_freq', 'stress_level_te', 'stress_level_freq', 'academic_work_impact_te', 'academic_work_im

In [5]:
# ===== Phase 2: native dtypes (numeric stays numeric, categorical stays category) + 10 engineered features, no te/freq =====
df_p2 = df.copy()
for col in categorical_cols:
    df_p2[col] = df_p2[col].astype("category")

df_p2 = engineer_features(df_p2)  # reuses Phase 1's engineer_features (same 10 engineered columns)

engineered_cols = [
    "missing_count", "total_breakdown_hours", "social_ratio", "gaming_ratio", "work_ratio",
    "unaccounted_screen_time", "screen_to_sleep_ratio", "weekend_vs_daily_ratio",
    "app_opens_per_hour", "notifications_per_hour",
]

feature_cols_phase2 = numeric_cols + categorical_cols + engineered_cols
X_phase2 = df_p2[feature_cols_phase2].copy()
categorical_cols_phase2 = categorical_cols  # only the 3 nominal columns stay category

print(f"Phase 2 features ({len(feature_cols_phase2)}):", feature_cols_phase2)

Phase 2 features (22): ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact', 'missing_count', 'total_breakdown_hours', 'social_ratio', 'gaming_ratio', 'work_ratio', 'unaccounted_screen_time', 'screen_to_sleep_ratio', 'weekend_vs_daily_ratio', 'app_opens_per_hour', 'notifications_per_hour']


In [6]:
# ===== Search machinery (shared by both phases) =====
def make_objective(X, y, categorical_cols, seed, n_splits):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 3000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "min_child_weight": trial.suggest_float("min_child_weight", 1, 12),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 100.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        }

        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        fold_scores = []

        for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

            common = dict(
                **params,
                tree_method="hist",
                enable_categorical=len(categorical_cols) > 0,
                eval_metric="auc",
                early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                random_state=seed,
                n_jobs=-1,
            )
            try:
                model = XGBClassifier(device="cuda", **common)
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            except Exception:
                model = XGBClassifier(device="cpu", **common)
                model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

            pred = model.predict_proba(X_va)[:, 1]
            fold_scores.append(roc_auc_score(y_va, pred))

            trial.report(float(np.mean(fold_scores)), step=fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(fold_scores))
    return objective


def run_search(X, y, categorical_cols, max_seconds, out_prefix, seed=SEED, n_splits=N_SPLITS):
    csv_path = os.path.join(out_dir, f"{out_prefix}_results.csv")
    json_path = os.path.join(out_dir, f"{out_prefix}_best.json")

    def save_callback(study, _trial):
        rows = [
            {"trial": t.number, "value": t.value, "state": str(t.state), **t.params}
            for t in study.trials if t.value is not None
        ]
        if rows:
            pd.DataFrame(rows).sort_values("value", ascending=False).to_csv(csv_path, index=False)
        if study.best_trial is not None:
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump({"best_value": study.best_value, "best_params": study.best_params}, f, indent=2)

    sampler = optuna.samplers.TPESampler(seed=seed)
    pruner = optuna.pruners.MedianPruner(n_warmup_steps=1)
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)

    print(f"Starting search '{out_prefix}' | time budget: {max_seconds/3600:.2f}h ({max_seconds:.0f}s)")
    start = time.perf_counter()
    study.optimize(
        make_objective(X, y, categorical_cols, seed, n_splits),
        timeout=max_seconds,
        callbacks=[save_callback],
        catch=(Exception,),
    )
    elapsed = time.perf_counter() - start
    save_callback(study, None)

    print(f"'{out_prefix}' done | trials={len(study.trials)} | best AUC={study.best_value:.4f} | elapsed={elapsed/60:.1f} min")
    print("Best params:", study.best_params)
    print("Saved results CSV:", csv_path)
    print("Saved best JSON:", json_path)
    return study

In [7]:
study_phase1 = run_search(X_phase1, y, categorical_cols_phase1, MAX_SECONDS_PHASE1, "phase1_full_tmp_pipeline")

Starting search 'phase1_full_tmp_pipeline' | time budget: 0.03h (120s)


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [20:51:24] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [20:51:24] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [20:51:58] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [20:51:58] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [20:52:28] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [20:52:28] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round =

'phase1_full_tmp_pipeline' done | trials=2 | best AUC=0.9667 | elapsed=33.9 min
Best params: {'n_estimators': 2212, 'learning_rate': 0.010573268083515799, 'max_depth': 10, 'min_child_weight': 10.156869048804639, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'reg_alpha': 0.0008260808399079611, 'reg_lambda': 0.033205591037519584, 'gamma': 2.6237821581611893}
Saved results CSV: /workspaces/Kaggle/KagglePredictPhoneAddiction/feature_eng_improvement_outputs/phase1_full_tmp_pipeline_results.csv
Saved best JSON: /workspaces/Kaggle/KagglePredictPhoneAddiction/feature_eng_improvement_outputs/phase1_full_tmp_pipeline_best.json


In [8]:
study_phase2 = run_search(X_phase2, y, categorical_cols_phase2, MAX_SECONDS_PHASE2, "phase2_native_types_engineered")

Starting search 'phase2_native_types_engineered' | time budget: 0.03h (120s)


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:25:17] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:25:17] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:26:13] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:26:13] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:27:15] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:27:15] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round =

'phase2_native_types_engineered' done | trials=1 | best AUC=0.9635 | elapsed=2.9 min
Best params: {'n_estimators': 1311, 'learning_rate': 0.13125830316209655, 'max_depth': 8, 'min_child_weight': 7.585243326167403, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 0.00019517224641449495, 'reg_lambda': 21.42302175774105, 'gamma': 3.005575058716044}
Saved results CSV: /workspaces/Kaggle/KagglePredictPhoneAddiction/feature_eng_improvement_outputs/phase2_native_types_engineered_results.csv
Saved best JSON: /workspaces/Kaggle/KagglePredictPhoneAddiction/feature_eng_improvement_outputs/phase2_native_types_engineered_best.json


In [9]:
print("=== Comparison ===")
print(f"Phase 1 (category + engineered ratios + te/freq) best CV AUC: {study_phase1.best_value:.4f}  ({len(study_phase1.trials)} trials)")
print(f"Phase 2 (native dtypes + engineered ratios only)  best CV AUC: {study_phase2.best_value:.4f}  ({len(study_phase2.trials)} trials)")

=== Comparison ===
Phase 1 (category + engineered ratios + te/freq) best CV AUC: 0.9667  (2 trials)
Phase 2 (native dtypes + engineered ratios only)  best CV AUC: 0.9635  (1 trials)
